In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import xarray as xr

from scipy.spatial import KDTree
import os

## Load generator data from PUDL 
https://data.catalyst.coop/preview/pudl/out_eia__yearly_generators

In [2]:
## Loading from PUDL (post-processed)
pudl = pd.read_csv(r"Generator data/Processed_PUDL_data_byLRZ.csv", index_col=0)


## Assign units to ungrouped LRZs
conditions = [
    (pudl["BA"] == "MISO-0001"),
    (pudl["BA"] == "MISO-0027") & (pudl["state"] == "WI"),
    (pudl["BA"] == "MISO-0027") & (pudl["state"] == "MI"),
    (pudl["BA"] == "MISO-0035") & pudl["state"].isin(["IA", "MN", "IL"]),
    (pudl["BA"] == "MISO-0035"),
    (pudl["BA"] == "MISO-0004"),
    (pudl["BA"] == "MISO-0006"),
    (pudl["BA"] == "MISO-8910") & (pudl["state"] == "AR"),
    (pudl["BA"] == "MISO-8910") & (pudl["state"].isin(["LA", "TX"])),
    (pudl["BA"] == "MISO-8910") & (pudl["state"] == "MS"),
]

choices = ["01", "02", "07", "03", "05", "04", "06", "08", "09", "10"]

pudl["LRZ"] = np.select(conditions, choices, default=pudl["BA"])
pudl

,Join_Count,TARGET_FID,plant_id_eia,generator_id,report_date,unit_id_pudl,plant_id_pudl,plant_name_eia,utility_id_eia,utility_id_pudl,...,ultrasupercritical_tech,uprate_derate_completed_date,uprate_derate_during_year,winter_capacity_estimate,winter_capacity_mw,winter_estimated_capability_mw,zip_code,MISO_LRZ,BA,LRZ
OID_,,,,,,,,,,,,,,,,,,,,,
1,1,1,30,1,1/1/2024 0:00,NaN,1272,Madelia,29305,1082,...,NaN,NaN,False,NaN,3.1,NaN,56062.0,NaN,MISO-0035,03
2,1,2,30,2,1/1/2024 0:00,NaN,1272,Madelia,29305,1082,...,NaN,NaN,False,NaN,2.0,NaN,56062.0,NaN,MISO-0035,03
3,1,3,30,3,1/1/2024 0:00,NaN,1272,Madelia,29305,1082,...,NaN,NaN,False,NaN,1.0,NaN,56062.0,NaN,MISO-0035,03
4,1,4,30,4,1/1/2024 0:00,NaN,1272,Madelia,29305,1082,...,NaN,NaN,False,NaN,4.0,NaN,56062.0,NaN,MISO-0035,03
5,1,5,30,5,1/1/2024 0:00,NaN,1272,Madelia,29305,1082,...,NaN,NaN,False,NaN,1.2,NaN,56062.0,NaN,MISO-0035,03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5122,1,5122,69063,APLS,1/1/2024 0:00,NaN,20426,Appleseed Solar,67237,17233,...,NaN,NaN,NaN,NaN,200.0,NaN,46947.0,NaN,MISO-0006,06
5123,1,5123,69066,11132,1/1/2024 0:00,NaN,20479,Goldenrod (IL),61194,6027,...,NaN,NaN,NaN,NaN,5.0,NaN,61486.0,NaN,MISO-0004,04
5124,1,5124,69070,11134,1/1/2024 0:00,NaN,20445,Anacott,61194,6027,...,NaN,NaN,NaN,NaN,5.0,NaN,62049.0,NaN,MISO-0004,04


## Create solar profiles

In [ ]:
solar_gen = pudl[pudl["fuel_type_code_pudl"]=="solar"]

## check all OIDs are unique:
if len(solar_gen.index.unique()) != len(solar_gen.index):
    print("WARNING: DUPLICATE UNIT IDS")

solar_gen = solar_gen[["plant_id_eia", "plant_name_eia", "capacity_mw", "generator_operating_date", "fuel_type_code_pudl", "latitude", "longitude", "LRZ"]]


In [ ]:
from pathlib import Path
year = 2015
CESM_BASE   = Path(rf"/gdex/data/d651009/b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002/atm/proc/tseries/hour_6")

fsds_in   = xr.open_dataset(CESM_BASE / rf"b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002.cam.h4.FSDS.{year}010100-{year+1}010100.nc")
u10_in    = xr.open_dataset(CESM_BASE / rf"b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002.cam.h2.U10.{year}010100-{year+1}010100.nc")
trefht_in = xr.open_dataset(CESM_BASE / rf"b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002.cam.h2.TREFHT.{year}010100-{year+1}010100.nc")

def calculate_cfs(fsds, u10, trefht):
    c1, c2, c3, c4 = 4.3, 0.943, 0.028, 1.528
    g, t_stc, rsds_stc = -0.005, 25, 1000
    cell_temp = c1 + c2*trefht + c3*fsds + c4*u10
    return (1 + g*(cell_temp - t_stc)) * (fsds/rsds_stc)

def assign_profiles(solar, nc_lat, nc_lon):
    """Returns full solar DataFrame with ncol_idx, cesm_lat, cesm_lon, profile_name added."""
    tree = KDTree(np.column_stack([nc_lat, nc_lon]))
    _, ncol_indices = tree.query(np.column_stack([solar["latitude"].values, solar["longitude"].values % 360]))

    solar = solar.copy()
    solar["ncol_idx"] = ncol_indices
    solar["cesm_lat"] = nc_lat[ncol_indices]
    solar["cesm_lon"] = nc_lon[ncol_indices]

    chunks = []
    for _, group in solar.groupby("LRZ"):
        pts = (group[["ncol_idx", "cesm_lat", "cesm_lon"]]
               .drop_duplicates("ncol_idx")
               .sort_values(["cesm_lat", "cesm_lon"], ascending=[False, True])
               .reset_index(drop=True))
        pts["grid_num"] = range(1, len(pts) + 1)
        chunks.append(group.merge(pts[["ncol_idx", "grid_num"]], on="ncol_idx", how="left"))
    solar = pd.concat(chunks, ignore_index=True)

    solar["profile_name"] = (
        "MISO_" + solar["LRZ"] + "_solar_"
        + solar["grid_num"].apply(lambda n: f"{int(n):03d}")
    )

    solar["profile_name"] = "MISO_" + solar["LRZ"] + "_solar_" + solar["grid_num"].apply(lambda n: f"{int(n):03d}")

    return solar

# Build mapping — keeps all original solar_gen columns
nc_lat = fsds_in["lat"].values
nc_lon = fsds_in["lon"].values
solar = solar_gen.dropna(subset=["latitude", "longitude"]).copy()
solar = assign_profiles(solar.reset_index(), nc_lat, nc_lon)

# solar.to_csv("solar_mapping.csv")

In [ ]:
solar.to_csv("Mapping/solar_map.csv")

### See generate_solar_profiles.py for full script

## Create Wind Profiles

In [3]:
wind_gen = pudl[pudl["fuel_type_code_pudl"]=="wind"]

## check all OIDs are unique:
if len(wind_gen.index.unique()) != len(wind_gen.index):
    print("WARNING: DUPLICATE UNIT IDS")

wind_gen = wind_gen[["plant_id_eia", "plant_name_eia", "capacity_mw", "generator_operating_date", "fuel_type_code_pudl", "latitude", "longitude", "LRZ"]]
wind_gen


,plant_id_eia,plant_name_eia,capacity_mw,generator_operating_date,fuel_type_code_pudl,latitude,longitude,LRZ
OID_,,,,,,,,
81,944,Geneseo,1.5,9/1/2009 0:00,wind,41.451492,-90.148550,04
82,944,Geneseo,1.5,10/1/2009 0:00,wind,41.451492,-90.148550,04
369,1172,Osage (IA),1.6,12/1/2009 0:00,wind,43.279720,-92.810555,03
946,1998,Mountain Lake,1.2,7/1/2007 0:00,wind,43.940500,-94.943400,03
1006,2022,Willmar,2.0,8/1/2009 0:00,wind,45.121704,-95.053240,01
...,...,...,...,...,...,...,...,...
5015,68413,Old Town Wind,178.0,NaN,wind,40.988976,-89.173260,04
5072,68843,Tupper Lake Wind,200.0,NaN,wind,42.829000,-85.138000,07
5075,68872,Shenandoah Hills Wind Farm,214.3,12/1/2025 0:00,wind,40.634110,-95.370700,03


In [4]:
# Check capacity-weighted average IEC wind class by zone (EIA reporting)
# Use the nearest class for each zone

wind_details = pd.read_excel(r"./Generator data/3_2_Wind_Y2024.xlsx", header=1)
wind_details = wind_details.rename(columns={"Plant Code":"plant_id_eia"})
wind_by_class = wind_gen.merge(wind_details[["plant_id_eia", "Wind Quality Class"]], how="left", on="plant_id_eia")[["LRZ", "capacity_mw", "Wind Quality Class"]].dropna(subset=["LRZ", "capacity_mw", "Wind Quality Class"])
wind_by_class.groupby("LRZ").apply(
    lambda g: np.average(g["Wind Quality Class"], weights=g["capacity_mw"])
).round()


LRZ
01    2.0
02    2.0
03    3.0
04    2.0
05    2.0
06    2.0
07    3.0
10    3.0
dtype: float64

In [ ]:
year = 2015

# u10_in    = xr.open_dataset(rf"../../../../Large Ensembles\CESM HR\U10\b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002.cam.h2.U10.{year}010100-{year+1}010100.nc", chunks="auto")
# trefht_in = xr.open_dataset(rf"../../../../Large Ensembles\CESM HR\TREFHT\b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002.cam.h2.TREFHT.{year}010100-{year+1}010100.nc", chunks="auto")
# qrefht_in = xr.open_dataset(rf"../../../../Large Ensembles\CESM HR\QREFHT\b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002.cam.h4.QREFHT.{year}010100-{year+1}010100.nc", chunks="auto")
# ps_in = xr.open_dataset(rf"../../../../Large Ensembles\CESM HR\PS\b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002.cam.h2.PS.{year}010100-{year+1}010100.nc", chunks="auto")
power_curve = pd.read_csv(r"./Generator data/wind_power_curve_class2.csv")


def calculate_wind_cfs(trefht, qrefht, u10, ps, power_curve):
    r = 287.05
    pd = ps/(r*trefht)
    pm = pd*(1+qrefht)/(1+1.609*qrefht)

    w100_p = u10*100**(1/7)/10

    w100 = w100_p*(pm/1.255)**(1/3)

    return np.interp(w100, power_curve["Wind Speed (m/s)"], power_curve["Turbine Output"])


def assign_profiles(wind, nc_lat, nc_lon):
    """Returns full solar DataFrame with ncol_idx, cesm_lat, cesm_lon, profile_name added."""
    tree = KDTree(np.column_stack([nc_lat, nc_lon]))
    _, ncol_indices = tree.query(np.column_stack([wind["latitude"].values, wind["longitude"].values % 360]))

    wind = wind.copy()
    wind["ncol_idx"] = ncol_indices
    wind["cesm_lat"] = nc_lat[ncol_indices]
    wind["cesm_lon"] = nc_lon[ncol_indices]


    chunks = []
    for _, group in wind.groupby("LRZ"):
        pts = (group[["ncol_idx", "cesm_lat", "cesm_lon"]]
               .drop_duplicates("ncol_idx")
               .sort_values(["cesm_lat", "cesm_lon"], ascending=[False, True])
               .reset_index(drop=True))
        pts["grid_num"] = range(1, len(pts) + 1)
        chunks.append(group.merge(pts[["ncol_idx", "grid_num"]], on="ncol_idx", how="left"))
    wind = pd.concat(chunks, ignore_index=True)
    
    wind["profile_name"] = "MISO_" + wind["LRZ"] + "_wind_" + wind["grid_num"].apply(lambda n: f"{int(n):03d}")
    return wind



# Build mapping 
nc_lat = trefht_in["lat"].values
nc_lon = trefht_in["lon"].values
wind = wind_gen.dropna(subset=["latitude", "longitude"]).copy()
wind = assign_profiles(wind.reset_index(), nc_lat, nc_lon)

unique_ncols  = np.unique(wind["ncol_idx"].values).tolist()
ncol_to_pos   = {int(nc): pos for pos, nc in enumerate(unique_ncols)}
profiles_list = list(zip(wind["profile_name"], wind["ncol_idx"].tolist()))


# time_vals  = trefht_in["time"].values
# .chunk() only on the float data variables — avoids object-dtype time coord issue
# qrefht_mem   = qrefht_in["QREFHT"].chunk({"time": 43}).isel(ncol=unique_ncols).compute().values
# u10_mem    = u10_in["U10"].chunk({"time": 43}).isel(ncol=unique_ncols).compute().values
# trefht_mem = trefht_in["TREFHT"].chunk({"time": 43}).isel(ncol=unique_ncols).compute().values - 273.15
# ps_mem = ps_in["PS"].chunk({"time": 43}).isel(ncol=unique_ncols).compute().values




In [ ]:
wind.to_csv("Mapping/wind_map.csv")

## Create thermal derate profiles

In [ ]:
thermal_list = ["Natural Gas Fired Combustion Turbine", "Natural Gas Steam Turbine", "Natural Gas Fired Combined Cycle", "Conventional Steam Coal"]

thermal_gen = pudl[pudl["technology_description"].isin(thermal_list)]
## check all OIDs are unique:
if len(thermal_gen.index.unique()) != len(thermal_gen.index):
    print("WARNING: DUPLICATE UNIT IDS")

thermal_gen = thermal_gen[["plant_id_eia", "plant_name_eia", "capacity_mw", "generator_operating_date", "technology_description", "latitude", "longitude", "LRZ"]]

# Load cooling type from EIA 860 6.2
cooling_info = pd.read_parquet(r"Generator data/_core_eia860__cooling_equipment.parquet")
thermal_gen = thermal_gen.reset_index().merge(cooling_info[["plant_id_eia", "cooling_type_1"]], how="left", on="plant_id_eia").drop_duplicates(subset=["OID_"])
thermal_gen = thermal_gen.set_index("OID_")

# Assume that any units that are not DC are RC
thermal_gen["cooling_type_1"] = thermal_gen["cooling_type_1"].fillna("RC")
thermal_gen["cooling_type_1"] = np.select([(thermal_gen["cooling_type_1"] != "DC"), (thermal_gen["cooling_type_1"] == "DC")], ["RC", "DC"], default=thermal_gen["cooling_type_1"])
thermal_gen

In [ ]:
year = 2015

# u10_in    = xr.open_dataset(rf"../../../../Large Ensembles\CESM HR\U10\b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002.cam.h2.U10.{year}010100-{year+1}010100.nc", chunks="auto")
# trefht_in = xr.open_dataset(rf"../../../../Large Ensembles\CESM HR\TREFHT\b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002.cam.h2.TREFHT.{year}010100-{year+1}010100.nc", chunks="auto")
# qrefht_in = xr.open_dataset(rf"../../../../Large Ensembles\CESM HR\QREFHT\b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002.cam.h4.QREFHT.{year}010100-{year+1}010100.nc", chunks="auto")
# ps_in = xr.open_dataset(rf"../../../../Large Ensembles\CESM HR\PS\b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002.cam.h2.PS.{year}010100-{year+1}010100.nc", chunks="auto")

def frac_capacity(tech, cooling, trefht, qrefht, ps):
    es = 6.107*100*np.exp(17.27*(trefht-273.15)/(trefht-35.85))
    e = 0.622
    qsat = e*es/(ps-(1-e)*es)
    rh = qrefht/qsat*100
    trefht = (trefht-273.15)*9/5+32 # convert to F

    if tech in ["Natural Gas Steam Turbine", "Conventional Steam Coal"]:
        beta_T = 0.0107
        beta_RH = 0.0287
        beta_TRH = -0.000378
        beta_P = 0.0315
        alpha = 0.195
    elif tech == "Natural Gas Fired Combined Cycle":
        beta_T = 0.0113
        beta_RH = 0.0266
        beta_TRH = -0.000336
        beta_P = 0.0161
        alpha = 0.12
    elif tech == "Natural Gas Fired Combustion Turbine":
        frac_avail_capacity = 0.0083*trefht + 1.15
        return frac_avail_capacity 
    else:
        raise ValueError
    
    if cooling == "RC":
        frac_avail_capacity = beta_T*trefht + beta_RH*rh + beta_TRH*(trefht*rh) + alpha
    elif cooling == "DC":
        frac_avail_capacity = beta_T*trefht + beta_P*ps + alpha
    else:
        raise TypeError
    return frac_avail_capacity



def assign_profiles(thermal, nc_lat, nc_lon):
    """Returns full thermal DataFrame with ncol_idx, cesm_lat, cesm_lon, profile_name added."""
    tree = KDTree(np.column_stack([nc_lat, nc_lon]))
    _, ncol_indices = tree.query(np.column_stack([thermal["latitude"].values, thermal["longitude"].values % 360]))

    thermal = thermal.copy()
    thermal["ncol_idx"] = ncol_indices
    thermal["cesm_lat"] = nc_lat[ncol_indices]
    thermal["cesm_lon"] = nc_lon[ncol_indices]


    chunks = []
    for _, group in thermal.groupby("LRZ"):
        pts = (group[["ncol_idx", "cesm_lat", "cesm_lon"]]
               .drop_duplicates("ncol_idx")
               .sort_values(["cesm_lat", "cesm_lon"], ascending=[False, True])
               .reset_index(drop=True))
        pts["grid_num"] = range(1, len(pts) + 1)
        chunks.append(group.merge(pts[["ncol_idx", "grid_num"]], on="ncol_idx", how="left"))
    thermal = pd.concat(chunks, ignore_index=True)
    thermal["profile_name"] = "MISO_" + thermal["LRZ"] + "_" + thermal["technology_description"].str.replace(" ", "_") + "_" + thermal["cooling_type_1"] + "_" + thermal["grid_num"].apply(lambda n: f"{int(n):03d}")
    return thermal



# Build mapping 
nc_lat = trefht_in["lat"].values
nc_lon = trefht_in["lon"].values
thermal = thermal_gen.dropna(subset=["latitude", "longitude"]).copy()
thermal = assign_profiles(thermal.reset_index(), nc_lat, nc_lon)

unique_ncols  = np.unique(thermal["ncol_idx"].values).tolist()
ncol_to_pos   = {int(nc): pos for pos, nc in enumerate(unique_ncols)}
profiles_list = list(zip(thermal["profile_name"], thermal["ncol_idx"].tolist(), thermal["technology_description"], thermal["cooling_type_1"]))


# time_vals  = trefht_in["time"].values
# # .chunk() only on the float data variables — avoids object-dtype time coord issue
# qrefht_mem   = qrefht_in["QREFHT"].chunk({"time": 43}).isel(ncol=unique_ncols).compute().values
# u10_mem    = u10_in["U10"].chunk({"time": 43}).isel(ncol=unique_ncols).compute().values
# trefht_mem = trefht_in["TREFHT"].chunk({"time": 43}).isel(ncol=unique_ncols).compute().values
# ps_mem = ps_in["PS"].chunk({"time": 43}).isel(ncol=unique_ncols).compute().values



# for profile_name, ncol_idx, tech, cooling in profiles_list:
#     pos = ncol_to_pos[ncol_idx]
#     cf = frac_capacity(tech, cooling, trefht_mem[:, pos], qrefht_mem[:, pos], ps_mem[:, pos])
#     pd.DataFrame({"time": time_vals, "cf": cf}).to_csv(
#     f"Thermal Derates/{profile_name}_{year}.csv", index=False
#     )


In [ ]:
thermal.to_csv("Mapping/thermal_derates_map.csv")

## Create Thermal FOR profiles

In [ ]:
thermal_list = ["Natural Gas Fired Combustion Turbine", 
                "Natural Gas Steam Turbine", 
                "Natural Gas Fired Combined Cycle", 
                "Conventional Steam Coal", 
                "Nuclear", 
                "Petroleum Liquids", 
                'Municipal Solid Waste',
                'Fossil Waste', 
                'Natural Gas Internal Combustion Engine', 
                'Non-Fossil Waste', 
                'Other Gases',
                'Biomass',
                'Other Waste Biomass', 
                'Wood/Wood Waste Biomass'
                "Landfill Gas",
                "Petroleum Coke", 
                "Coal Integrated Gasification Combined Cycle"
                ]

thermal_gen = pudl[pudl["technology_description"].isin(thermal_list)]
## check all OIDs are unique:
if len(thermal_gen.index.unique()) != len(thermal_gen.index):
    print("WARNING: DUPLICATE UNIT IDS")

thermal_gen = thermal_gen[["plant_id_eia", "plant_name_eia", "capacity_mw", "generator_operating_date", "fuel_type_code_pudl", "technology_description", "prime_mover_code", "latitude", "longitude", "LRZ"]]

# Align types with FOR table codes
tech_categories = {
    "CC" : ["Natural Gas Fired Combined Cycle", "Coal Integrated Gasification Combined Cycle"],
    "CT" : ["Natural Gas Fired Combustion Turbine", 'Natural Gas Internal Combustion Engine', "Landfill Gas", "Petroleum Liquids"],
    "ST" : ["Natural Gas Steam Turbine", "Conventional Steam Coal", 'Biomass', 'Other Waste Biomass', 'Wood/Wood Waste Biomass', "Petroleum Coke", 'Other Gases', 'Municipal Solid Waste',],
    "NU" : ["Nuclear"], 
}
for_table = pd.read_csv(r"./Generator data/temperature_dependent_outage_rates.csv")

tech_map = {tech: cat for cat, techs in tech_categories.items() for tech in techs}
thermal_gen["tech_category"] = thermal_gen["technology_description"].map(tech_map)
thermal_gen

In [ ]:
def assign_profiles(thermal, nc_lat, nc_lon):
    """Returns full thermal DataFrame with ncol_idx, cesm_lat, cesm_lon, profile_name added."""
    tree = KDTree(np.column_stack([nc_lat, nc_lon]))
    _, ncol_indices = tree.query(np.column_stack([thermal["latitude"].values, thermal["longitude"].values % 360]))

    thermal = thermal.copy()
    thermal["ncol_idx"] = ncol_indices
    thermal["cesm_lat"] = nc_lat[ncol_indices]
    thermal["cesm_lon"] = nc_lon[ncol_indices]

    chunks = []
    for _, group in thermal.groupby("LRZ"):
        pts = (group[["ncol_idx", "cesm_lat", "cesm_lon"]]
               .drop_duplicates("ncol_idx")
               .sort_values(["cesm_lat", "cesm_lon"], ascending=[False, True])
               .reset_index(drop=True))
        pts["grid_num"] = range(1, len(pts) + 1)
        chunks.append(group.merge(pts[["ncol_idx", "grid_num"]], on="ncol_idx", how="left"))
    thermal = pd.concat(chunks, ignore_index=True)

    thermal["profile_name"] = "MISO_" + thermal["LRZ"] + "_" + thermal["tech_category"].str.replace(" ", "_") + "_" + thermal["grid_num"].apply(lambda n: f"{int(n):03d}")
    return thermal

year=2015
trefht_in  = xr.open_dataset(f"/gdex/data/d651009/b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002/atm/proc/tseries/hour_6/b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002.cam.h2.TREFHT.{year}010100-{year+1}010100.nc")

# trefht_in = xr.open_dataset(rf"../../../../Large Ensembles\CESM HR\TREFHT\b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002.cam.h2.TREFHT.{year}010100-{year+1}010100.nc", chunks="auto")

# Build mapping 
nc_lat = trefht_in["lat"].values
nc_lon = trefht_in["lon"].values
thermal = thermal_gen.dropna(subset=["latitude", "longitude"]).copy()
thermal = assign_profiles(thermal.reset_index(), nc_lat, nc_lon)

unique_ncols  = np.unique(thermal["ncol_idx"].values).tolist()
ncol_to_pos   = {int(nc): pos for pos, nc in enumerate(unique_ncols)}
profiles_list = list(zip(thermal["profile_name"], thermal["ncol_idx"].tolist(), thermal["tech_category"]))

# time_vals  = trefht_in["time"].values
# # .chunk() only on the float data variables — avoids object-dtype time coord issue
# trefht_mem = trefht_in["TREFHT"].chunk({"time": 43}).isel(ncol=unique_ncols).compute().values -273.15

# def calculate_fors(tech, trefht):
#     trefht_rounded = np.clip(5 * np.round(trefht / 5), -15, 35).astype(int)
#     mapping = for_table.set_index("Temperature")[tech]
#     return mapping[trefht_rounded].values

# for profile_name, ncol_idx, tech in profiles_list:
#     pos = ncol_to_pos[ncol_idx]
#     outage_rate = calculate_fors(tech, trefht_mem[:, pos])
#     # pd.DataFrame({"time": time_vals, "for": outage_rate}).to_csv(
#     # f"Thermal FORs/{profile_name}_{year}.csv", index=False
#     # )


In [ ]:
thermal.to_csv("Mapping/thermal_fors_map.csv")

### Map resources to profiles

In [ ]:
## Loading from PUDL (post-processed)
pudl = pd.read_csv(r"Generator data/Processed_PUDL_data_byLRZ.csv", index_col=0)

solar_map = pd.read_csv("Mapping/solar_map.csv", index_col=1)
wind_map = pd.read_csv("Mapping/wind_map.csv", index_col=1)
thermal_derate_map = pd.read_csv("Mapping/thermal_derates_map.csv", index_col=1)

pudl = pudl.join(solar_map["profile_name"]).rename(columns={"profile_name" : "solar_profile_name"})
pudl = pudl.join(wind_map["profile_name"]).rename(columns={"profile_name" : "wind_profile_name"})
pudl = pudl.join(thermal_derate_map["profile_name"]).rename(columns={"profile_name" : "thermal_derate_name"})
pudl

In [ ]:
pudl.to_csv("Mapping/pudl_with_profile_names.csv")

## Unused: Benchmarking Generator stuff

In [ ]:
## Loading directly from EIA 860

def load_generator_data(existing:pd.DataFrame, planned:pd.DataFrame, plants: pd.DataFrame, ba:str):
    ids_ba = plants[plants["Balancing Authority Code"]==ba][["Plant Code", 'Latitude', 'Longitude']]
    
    existing = existing[existing["Plant Code"].isin(ids_ba["Plant Code"])]
    existing["Operational Status"] = "existing"

    planned = planned[planned["Plant Code"].isin(ids_ba["Plant Code"])]
    planned["Operational Status"] = "planned" 

    generators = pd.concat([existing.reset_index(drop=True), planned.reset_index(drop=True)], axis=0)
    generators = generators.merge(ids_ba.drop_duplicates(subset="Plant Code"), on="Plant Code", how="left")
    return generators



plants = pd.read_excel(r"Generator data\2___Plant_Y2024.xlsx", header =1)
existing_gen = pd.read_excel(r"Generator data\3_1_Generator_Y2024.xlsx", sheet_name="Operable", header=1)
planned_gen =  pd.read_excel(r"Generator data\3_1_Generator_Y2024.xlsx", sheet_name="Proposed", header=1)


miso_gen =load_generator_data(existing_gen, planned_gen, plants, "MISO")
miso_gen

In [ ]:
# Snap each plant to nearest 0.25-degree grid point
solar_gen["grid_lat"] = (solar_gen["latitude"] / 0.25).round() * 0.25
solar_gen["grid_lon"] = (solar_gen["longitude"] / 0.25).round() * 0.25

# Within each LRZ, number unique grid points N→S then W→E
def assign_grid_num(group):
    pts = (group[["grid_lat", "grid_lon"]]
           .drop_duplicates()
           .sort_values(["grid_lat", "grid_lon"], ascending=[False, True])
           .reset_index(drop=True))
    pts["grid_num"] = range(1, len(pts) + 1)
    return group.merge(pts, on=["grid_lat", "grid_lon"], how="left")

solar_gen = solar_gen.groupby("LRZ", group_keys=False).apply(assign_grid_num)

solar_gen["profile_name"] = "MISO_" + solar_gen["LRZ"] + "_solar_" + solar_gen["grid_num"].apply(lambda n: f"{int(n):03d}")
solar_gen = solar_gen.drop(columns=["grid_num"])

solar_gen[["latitude", "longitude", "grid_lat", "grid_lon", "LRZ", "profile_name"]].head(20)

In [ ]:
profile_name, ncol_idx, tech, cooling = profiles_list[0]

pos = ncol_to_pos[ncol_idx]
cf = frac_capacity(tech, cooling, trefht_mem[:, pos], qrefht_mem[:, pos], ps_mem[:, pos])

cf[676]
trefht = trefht_mem[:, pos][676]
qrefht = qrefht_mem[:, pos][676]
ps = ps_mem[:, pos][676]

es = 6.107*100*np.exp(17.27*(trefht-273.15)/(trefht-35.85))
e = 0.622
qsat = e*es/(ps-(1-e)*es)
rh = qrefht/qsat
trefht = (trefht-273.15)*9/5+32 # convert to F


beta_T = 0.0107
beta_RH = 0.0287
beta_TRH = -0.000378
beta_P = 0.0315
alpha = 0.195

In [ ]:
beta_T*trefht + beta_RH*rh + beta_TRH*(trefht*rh) + alpha
trefht

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

profile_name, ncol_idx, tech, cooling = profiles_list[0]
pos = ncol_to_pos[ncol_idx]

_trefht = trefht_mem[:, pos]
_qrefht = qrefht_mem[:, pos]
_ps     = ps_mem[:, pos]
cf      = frac_capacity(tech, cooling, _trefht, _qrefht, _ps)

# Derive RH for full time series (same formula as frac_capacity)
es   = 6.107 * 100 * np.exp(17.27 * (_trefht - 273.15) / (_trefht - 35.85))
e    = 0.622
qsat = e * es / (_ps - (1 - e) * es)
rh   = _qrefht / qsat          # fraction 0–1
temp_c = _trefht - 273.15

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    subplot_titles=("Temperature (°C)", "Relative Humidity", "Capacity Factor"),
)

fig.add_trace(go.Scatter(x=time_vals, y=temp_c, name="Temp",line=dict(color="tomato",   width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=time_vals, y=rh,     name="RH",  line=dict(color="steelblue",width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=time_vals, y=cf,      name="CF",  line=dict(color="seagreen", width=1)), row=3, col=1)

fig.update_yaxes(title_text="°C", row=1, col=1)
fig.update_yaxes(title_text="RH", row=2, col=1)
fig.update_yaxes(title_text="CF", row=3, col=1)
fig.update_layout(
    height=700,
    title_text=f"{profile_name}  |  {tech}  |  cooling: {cooling}",
    showlegend=False,
)
fig.show()


In [ ]:
import plotly.graph_objects as go

profile_name, ncol_idx, tech, cooling = profiles_list[0]
pos = ncol_to_pos[ncol_idx]

# ── Analytical contour grid ───────────────────────────────────────────────────
t_grid = np.linspace(20, 45, 300)
rh_grid = np.linspace(15, 100, 300)
T, RH = np.meshgrid(t_grid, rh_grid)
temp_f = T * 9/5 + 32

if tech in ["Natural Gas Steam Turbine", "Conventional Steam Coal"]:
    beta_T, beta_RH, beta_TRH, alpha = 0.0107, 0.0287, -0.000378, 0.195
elif tech == "Natural Gas Fired Combined Cycle":
    beta_T, beta_RH, beta_TRH, alpha = 0.0113, 0.0266, -0.000336, 0.12
elif tech == "Natural Gas Fired Combustion Turbine":
    beta_T, beta_RH, beta_TRH, alpha = 0.0083, 0.0, 0.0, 1.15

cf_grid = np.clip(beta_T * temp_f + beta_RH * RH + beta_TRH * (temp_f * RH) + alpha, 0, 1)

# ── Per-timestep data points ──────────────────────────────────────────────────
_trefht = trefht_mem[:, pos]          # K
_qrefht = qrefht_mem[:, pos]
_ps     = ps_mem[:, pos]
_cf     = frac_capacity(tech, cooling, _trefht, _qrefht, _ps)

_es   = 6.107 * 100 * np.exp(17.27 * (_trefht - 273.15) / (_trefht - 35.85))
_e    = 0.622
_qsat = _e * _es / (_ps - (1 - _e) * _es)
_rh_pct  = (_qrefht / _qsat) * 100       # → %
_temp_c  = _trefht - 273.15

# ── Plot ─────────────────────────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Contour(
    x=t_grid, y=rh_grid, z=cf_grid,
    colorscale="RdYlGn", zmin=0, zmax=1,
    ncontours=20,
    colorbar=dict(title="Fraction<br>Capacity<br>Available", thickness=15),
    contours=dict(showlabels=False),
    name="Model",
))

fig.add_trace(go.Scatter(
    x=_temp_c, y=_rh_pct,
    mode="markers",
    marker=dict(
        color=_cf,
        colorscale="RdYlGn", cmin=0, cmax=1,
        size=4,
        line=dict(width=0.3, color="black"),
        showscale=False,
    ),
    text=[str(t)[:16] for t in time_vals],
    hovertemplate="<b>%{text}</b><br>Temp: %{x:.1f}°C<br>RH: %{y:.1f}%<br>CF: %{marker.color:.3f}<extra></extra>",
    name="Hourly data",
))

fig.update_layout(
    title=f"{tech.replace('Natural Gas ', 'NG ')} | {cooling} cooling",
    xaxis_title="Air Temperature (°C)",
    yaxis_title="Relative Humidity (%)",
    width=620, height=500,
    legend=dict(x=0.01, y=0.01),
)
fig.show()


## Geopandas maps of FSDS (surface solar radiation)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import pandas as pd
from pathlib import Path
import cartopy.io.shapereader as shpreader

# ── Spatial coords (lon is 0-360 in CESM, convert to -180/180) ───────────────
lat     = fsds_in["lat"].values
lon     = fsds_in["lon"].values
lon_180 = np.where(lon > 180, lon - 360, lon)

# Annual-mean FSDS (W m⁻²) — takes ~30s on a full year of 6-hourly data
fsds_mean = fsds_in["FSDS"].mean(dim="time").values

# ── Clip to MISO bounding box ─────────────────────────────────────────────────
mask = (lat >= 25) & (lat <= 52) & (lon_180 >= -105) & (lon_180 <= -75)
lat_s, lon_s, fsds_s = lat[mask], lon_180[mask], fsds_mean[mask]

# ── US state + country boundaries via cartopy natural-earth cache ─────────────
states_path   = shpreader.natural_earth(resolution="10m", category="cultural",
                                        name="admin_1_states_provinces_lakes")
countries_path = shpreader.natural_earth(resolution="50m", category="cultural",
                                         name="admin_0_countries")
bbox = (-105, 25, -75, 52)
states    = gpd.read_file(states_path).clip(bbox)
states    = states[states["admin"] == "United States of America"]
countries = gpd.read_file(countries_path).clip(bbox)

# ── Delaunay triangulation — mask edges that span > 2° (grid-edge artefacts) ──
triang = mtri.Triangulation(lon_s, lat_s)
tri = triang.triangles
edge_max = np.max([
    np.hypot(lon_s[tri[:,0]]-lon_s[tri[:,1]], lat_s[tri[:,0]]-lat_s[tri[:,1]]),
    np.hypot(lon_s[tri[:,1]]-lon_s[tri[:,2]], lat_s[tri[:,1]]-lat_s[tri[:,2]]),
    np.hypot(lon_s[tri[:,2]]-lon_s[tri[:,0]], lat_s[tri[:,2]]-lat_s[tri[:,0]]),
], axis=0)
triang.set_mask(edge_max > 2.0)

# ── Map 1: annual mean ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 8))

tp = ax.tripcolor(triang, fsds_s, cmap="YlOrRd", shading="gouraud", vmin=100, vmax=280)
plt.colorbar(tp, ax=ax, label="Mean FSDS (W m⁻²)", shrink=0.75, pad=0.02)

countries.boundary.plot(ax=ax, color="black", linewidth=1.0, zorder=3)
states.boundary.plot(ax=ax, color="gray", linewidth=0.4, zorder=3)

ax.set_xlim(-105, -75); ax.set_ylim(25, 52)
ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
ax.set_title(f"CESM Annual-Mean Surface Solar Radiation (FSDS) — {year}", fontsize=13)
ax.set_aspect("equal")
plt.tight_layout()
Path("Maps").mkdir(exist_ok=True)
plt.savefig(f"Maps/fsds_annual_mean_{year}.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Map 2: summer-solstice noon snapshot ──────────────────────────────────────
times = pd.DatetimeIndex(fsds_in["time"].values)
# pick the timestep on ~June 21 with highest global FSDS in MISO
june_mask = (times.month == 6) & (times.day >= 19) & (times.day <= 23)
june_idx  = np.where(june_mask)[0]
# choose the 6-hourly step with maximum mean FSDS over the domain
best_t = june_idx[np.argmax(fsds_in["FSDS"].isel(time=june_idx, ncol=mask.nonzero()[0]).mean(dim="ncol").values)]

fsds_snap = fsds_in["FSDS"].isel(time=best_t).values[mask]
t_str = str(times[best_t])[:16]

fig2, ax2 = plt.subplots(figsize=(12, 8))
triang2 = mtri.Triangulation(lon_s, lat_s)
triang2.set_mask(edge_max > 2.0)

tp2 = ax2.tripcolor(triang2, fsds_snap, cmap="YlOrRd", shading="gouraud", vmin=0, vmax=950)
plt.colorbar(tp2, ax=ax2, label="FSDS (W m⁻²)", shrink=0.75, pad=0.02)

countries.boundary.plot(ax=ax2, color="black", linewidth=1.0, zorder=3)
states.boundary.plot(ax=ax2, color="gray", linewidth=0.4, zorder=3)

ax2.set_xlim(-105, -75); ax2.set_ylim(25, 52)
ax2.set_xlabel("Longitude (°E)"); ax2.set_ylabel("Latitude (°N)")
ax2.set_title(f"CESM FSDS Snapshot — {t_str}", fontsize=13)
ax2.set_aspect("equal")
plt.tight_layout()
plt.savefig(f"Maps/fsds_snapshot_{year}.png", dpi=150, bbox_inches="tight")
plt.show()


## Geopandas maps of U10 (10-m wind speed)

In [ ]:
lat_u    = u10_in["lat"].values
lon_u    = u10_in["lon"].values
lon_u180 = np.where(lon_u > 180, lon_u - 360, lon_u)

u10_mean = u10_in["U10"].mean(dim="time").values

# ── Clip to MISO bounding box ─────────────────────────────────────────────────
mask_u = (lat_u >= 25) & (lat_u <= 52) & (lon_u180 >= -105) & (lon_u180 <= -75)
lat_us, lon_us, u10_s = lat_u[mask_u], lon_u180[mask_u], u10_mean[mask_u]

# ── Triangulation (reuse same edge-masking approach) ──────────────────────────
triang_u = mtri.Triangulation(lon_us, lat_us)
tri_u    = triang_u.triangles
edge_max_u = np.max([
    np.hypot(lon_us[tri_u[:,0]]-lon_us[tri_u[:,1]], lat_us[tri_u[:,0]]-lat_us[tri_u[:,1]]),
    np.hypot(lon_us[tri_u[:,1]]-lon_us[tri_u[:,2]], lat_us[tri_u[:,1]]-lat_us[tri_u[:,2]]),
    np.hypot(lon_us[tri_u[:,2]]-lon_us[tri_u[:,0]], lat_us[tri_u[:,2]]-lat_us[tri_u[:,0]]),
], axis=0)
triang_u.set_mask(edge_max_u > 2.0)

# ── Map 1: annual mean U10 ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 8))

tp = ax.tripcolor(triang_u, u10_s, cmap="Blues", shading="gouraud", vmin=0, vmax=12)
plt.colorbar(tp, ax=ax, label="Mean U10 (m s⁻¹)", shrink=0.75, pad=0.02)

countries.boundary.plot(ax=ax, color="black", linewidth=1.0, zorder=3)
states.boundary.plot(ax=ax, color="gray", linewidth=0.4, zorder=3)

ax.set_xlim(-105, -75); ax.set_ylim(25, 52)
ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
ax.set_title(f"CESM Annual-Mean 10-m Wind Speed (U10) — {year}", fontsize=13)
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(f"Maps/u10_annual_mean_{year}.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Map 2: winter snapshot (Jan peak wind period) ─────────────────────────────
times_u = pd.DatetimeIndex(u10_in["time"].values)
jan_mask = (times_u.month == 1)
jan_idx  = np.where(jan_mask)[0]
best_t_u = jan_idx[np.argmax(u10_in["U10"].isel(time=jan_idx, ncol=mask_u.nonzero()[0]).mean(dim="ncol").values)]

u10_snap = u10_in["U10"].isel(time=best_t_u).values[mask_u]
t_str_u  = str(times_u[best_t_u])[:16]

fig2, ax2 = plt.subplots(figsize=(12, 8))
triang_u2 = mtri.Triangulation(lon_us, lat_us)
triang_u2.set_mask(edge_max_u > 2.0)

tp2 = ax2.tripcolor(triang_u2, u10_snap, cmap="Blues", shading="gouraud", vmin=0, vmax=20)
plt.colorbar(tp2, ax=ax2, label="U10 (m s⁻¹)", shrink=0.75, pad=0.02)

countries.boundary.plot(ax=ax2, color="black", linewidth=1.0, zorder=3)
states.boundary.plot(ax=ax2, color="gray", linewidth=0.4, zorder=3)

ax2.set_xlim(-105, -75); ax2.set_ylim(25, 52)
ax2.set_xlabel("Longitude (°E)"); ax2.set_ylabel("Latitude (°N)")
ax2.set_title(f"CESM U10 Snapshot (peak Jan wind) — {t_str_u}", fontsize=13)
ax2.set_aspect("equal")
plt.tight_layout()
plt.savefig(f"Maps/u10_snapshot_{year}.png", dpi=150, bbox_inches="tight")
plt.show()


## Compare to ERA5

In [ ]:
import xarray as xr

year = 2015
u10_cesm = xr.open_dataset(f"/gdex/data/d651009/b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002/atm/proc/tseries/hour_6/b.e13.BRCP85C5.ne120_t12.cesm-ihesp-hires1.0.30-2006-2100.002.cam.h2.U10.{year}010100-{year+1}010100.nc")
u10_era5 = xr.open_dataset(f"/gdex/data/d633000/e5.oper.an.sfc/201001/e5.oper.an.sfc.128_165_10u.ll025sc.{year}010100_{year}013123.nc", auto="chunks")
